In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
import requests
from langchain.tools import tool
from dotenv import load_dotenv
import os

/home/saqib-mehdi/Desktop/LangChain (Generative AI) - CampusX/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# tool for currency converter factor

@tool
def get_currency_conversion_rate(from_currency: str, to_currency: str) -> float:
    """Get the conversion rate from one currency to another using an API."""

    url = f"https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{from_currency}/{to_currency}"
    response = requests.get(url)
    data = response.json()
    return data

# tool for multiplication of current currency value with the target conversion rate

@tool
def convert_currency_amount(base_currency_value: int, conversion_rate: float) -> float:
    """Convert the base currency value to target currency using conversion rate."""
    converted_amount = base_currency_value * conversion_rate
    return converted_amount

In [13]:
get_currency_conversion_rate.invoke({"from_currency": "INR", "to_currency": "PKR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1762819201,
 'time_last_update_utc': 'Tue, 11 Nov 2025 00:00:01 +0000',
 'time_next_update_unix': 1762905601,
 'time_next_update_utc': 'Wed, 12 Nov 2025 00:00:01 +0000',
 'base_code': 'INR',
 'target_code': 'PKR',
 'conversion_rate': 3.188}

In [14]:
conversion_rate = get_currency_conversion_rate.invoke({"from_currency": "INR", "to_currency": "PKR"})['conversion_rate']

converted_amount = convert_currency_amount.invoke({"base_currency_value": 1000, "conversion_rate": conversion_rate})

print(f"Converted Amount: {converted_amount} PKR")

Converted Amount: 3188.0 PKR


In [20]:
# tool binding

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

llm_with_tools = llm.bind_tools([get_currency_conversion_rate, convert_currency_amount])

In [21]:

# tool calling

messages = [
    HumanMessage('What is the conversion of 1000 PKR to USD?')
]

messages

[HumanMessage(content='What is the conversion of 1000 PKR to USD?', additional_kwargs={}, response_metadata={})]

In [22]:
result = llm_with_tools.invoke(messages)
result.tool_calls

[{'name': 'get_currency_conversion_rate',
  'args': {'to_currency': 'USD', 'from_currency': 'PKR'},
  'id': 'b419e004-c30c-44b0-aef3-0d2659b78b53',
  'type': 'tool_call'}]